draw EECs and ratios for injected v2 plots

In [2]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os
import ctypes
import math
import array

In [3]:
high_bins = [ [60,71], [71,78], [78,91], [91,97], [97,1000] ]
inclusive_bins = [ [0,25], [25,36], [36,48], [48,60]  ]
analysis_bins = [ [0,25], [25,36], [36,48], [48,60], [60,71], [71,78], [78,91], [91,97], [97,1000] ]

In [4]:
ROOT.gDirectory.Clear()

# inclusive filepaths
filepath_0mb_inc = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_inclusive/EEC_non_binned_Output_Batch0.root"
filepath_3mb_inc = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/3mb_inclusive/EEC_non_binned_Output_Batch0.root"
filepath_0mb_01_v2 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_inclusive/injected_v2_0_1/EEC_non_binned_Output_Batch0.root"
filepath_0mb_03_v2 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_inclusive/injected_v2_0_3/EEC_non_binned_Output_Batch0.root"

# high nch filepaths:
filepath_3mb_nch60 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/3mb_nch60/Merged_EECs.root"
filepath_0mb_nch60 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_nch60/Merged_EECs.root"
filepath_v2_nch60_01 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_nch60/injected_v2_01/Merged_EECs.root"
filepath_v2_nch60_03 = "/Users/rohanjagadeesan/Desktop/Code/li_lab/EnergyCorrelators/output/0mb_nch60/injected_v2_03/Merged_EECs.root"


In [5]:
# get and normalise the histograms

def initialise(filepath1, filepath2):
    '''
    returns normalised EECs
    '''

    tfile1 = ROOT.TFile.Open(filepath1, "READ")
    num_jets1 = tfile1.Get("num_jets_STD").GetVal()
    EEC1 = tfile1.Get("hEEC_STD")
    EEC1.SetDirectory(0)
    EEC1.Scale(1/num_jets1)
    tfile1.Close()

    tfile2 = ROOT.TFile.Open(filepath2, "READ")
    num_jets2 = tfile2.Get("num_jets_STD").GetVal()
    EEC2 = tfile2.Get("hEEC_STD")
    EEC2.SetDirectory(0)
    EEC2.Scale(1/num_jets2)
    tfile2.Close()

    return EEC1, EEC2

In [6]:

def initialise_highmult(filepath):
    tfile = ROOT.TFile.Open(filepath, "READ")
    EECs = {} # using standard

    for mult_bin in high_bins:
        bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
        bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

        #wta_profiles[bin_key] = ROOT.TGraphErrors( tfile.Get(f"WTA_profile_{bin_name}").Clone() )
        #std_profiles[bin_key] = ROOT.TGraphErrors( tfile.Get(f"STD_profile_{bin_name}").Clone() )
        #EECs[bin_key] = ROOT.TGraphErrors( tfile.Get(f"STD_eec_{bin_name}").Clone() )

        EECs[bin_key] = tfile.Get(f"STD_eec_{bin_name}").Clone()

        # detach them
        #wta_profiles[bin_key].SetDirectory(0)
        #std_profiles[bin_key].SetDirectory(0)
        EECs[bin_key].SetDirectory(0)
    
    #return wta_profiles, std_profiles, EECs
    tfile.Close()
    
    return EECs

In [7]:
def rebin_log(hist, name, num_bins_target=50):
    '''
    input: 
    - the ratio histogram
    - the name of the new rebinned histograms
    - the desired number of logarithmic bins
    
    output: rebinned log-spaced versions of the histograms
    '''
    import array
    
    # 1. Define the limits (your histograms range from 0 to 1)
    # Since log(0) is undefined, start slightly above 0 matching your plot limits (e.g., 10^-3)
    xmin = 0.001
    xmax = 1.0
    
    # 2. Generate log-spaced bin edges using numpy
    # logspace arguments are the exponents: 10^-3 to 10^0
    bin_edges_np = np.logspace(math.log10(xmin), math.log10(xmax), num_bins_target + 1)
    
    # 3. Convert the numpy array to a standard Python/C-compatible array for ROOT
    bin_edges_array = array.array('d', bin_edges_np)
    
    # 4. Create new empty histograms with variable bin widths
    new_hist = ROOT.TH1D(name, f"{hist.GetTitle()};#Delta R;EEC", num_bins_target, bin_edges_array)
    
    # Ensure Sumw2 is enabled to compute correct errors for variable bin widths
    new_hist.Sumw2()
    
    # 5. Manually remap the bin contents from the old fine linear bins to the new log bins
    # We loop over the fine uniform bins of the original histogram
    for old_bin in range(1, hist.GetNbinsX() + 1):
        bin_center = hist.GetBinCenter(old_bin)
        
        # Skip values below our log starting threshold to prevent underflow clutter
        if bin_center < xmin:
            continue
            
        bin_content = hist.GetBinContent(old_bin)
        bin_error = hist.GetBinError(old_bin)
        
        # Find which log-bin this center falls into
        new_bin = new_hist.FindBin(bin_center)
        
        # Add content and propagate errors in quadrature
        new_hist.SetBinContent(new_bin, new_hist.GetBinContent(new_bin) + bin_content)
        new_hist.SetBinError(new_bin, math.sqrt(new_hist.GetBinError(new_bin)**2 + bin_error**2))
        
    return new_hist

In [8]:
colours = [ROOT.kBlue+1, ROOT.kMagenta+1, ROOT.kSpring+4, ROOT.kRed+1, ROOT.kOrange+7]

# inclusive

## EEC plots

In [22]:
# log version, with markers

ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)
eec_03_v2 , eec_01_v2 = initialise(filepath_0mb_03_v2, filepath_0mb_01_v2)

# make TGraphErrors
tgraph_eec_0mb = ROOT.TGraphErrors(eec_0mb)
tgraph_eec_3mb = ROOT.TGraphErrors(eec_3mb)
tgraph_eec_v2 = ROOT.TGraphErrors(eec_01_v2)
tgraph_eec_v2_03 = ROOT.TGraphErrors(eec_03_v2)

# Choosing colours:
tgraph_eec_0mb.SetLineColorAlpha(colours[0], 0.5)
tgraph_eec_3mb.SetLineColorAlpha(colours[4], 0.5)
tgraph_eec_v2.SetLineColorAlpha(colours[2], 0.5)
tgraph_eec_v2_03.SetLineColorAlpha(colours[3], 0.5)

tgraph_eec_0mb.SetMarkerColorAlpha(colours[0], 0.5)
tgraph_eec_3mb.SetMarkerColorAlpha(colours[4], 0.5)
tgraph_eec_v2.SetMarkerColorAlpha(colours[2], 0.5)
tgraph_eec_v2_03.SetMarkerColorAlpha(colours[3], 0.5)

# markers:
tgraph_eec_0mb.SetMarkerStyle(20)
tgraph_eec_3mb.SetMarkerStyle(20)
tgraph_eec_v2.SetMarkerStyle(20)
tgraph_eec_v2_03.SetMarkerStyle(20)

tgraph_eec_0mb.SetMarkerSize(0.4)
tgraph_eec_3mb.SetMarkerSize(0.4)
tgraph_eec_v2.SetMarkerSize(0.4)
tgraph_eec_v2_03.SetMarkerSize(0.4)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l", "log EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_eec_0mb, "PE")
mg.Add(tgraph_eec_3mb, "PE")
mg.Add(tgraph_eec_v2, "PE")
mg.Add(tgraph_eec_v2_03, "PE")

# title
mg.SetTitle("EEC for inclusive datasets; #Delta R_{L}; E2C")
mg.Draw("A") 

# Log scale axes
canvas.SetLogx(1)
canvas.SetLogy(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0.0001, 1.0)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)

# Legend setup
legend = ROOT.TLegend(0.28, 0.28, 0.68, 0.48) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_eec_0mb, "0mb", "PE")
legend.AddEntry(tgraph_eec_3mb, "3.0mb", "PE")
legend.AddEntry(tgraph_eec_v2, "v_{2}=0.1", "PE")
legend.AddEntry(tgraph_eec_v2_03, "v_{2}=0.3", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_inclusive_log_plot_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_inclusive_log_plot_markers_v2.pdf has been created


## ratios

In [11]:
# ratio, with markers
# log x axis, rebinned
# plot injected over 0mb and 3mb over 0mb ratios


ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)

eec_03_v2 , eec_01_v2 = initialise(filepath_0mb_03_v2, filepath_0mb_01_v2)  

#rebin
rebinned_eec_0mb = rebin_log(eec_0mb, "eec_0mb_log", 50)
rebinned_eec_3mb = rebin_log(eec_3mb, "eec_3mb_log", 50)
rebinned_eec_01_v2 = rebin_log(eec_01_v2, "eec_01_v2_log", 50)
rebinned_eec_03_v2 = rebin_log(eec_03_v2, "eec_03_v2_log", 50)

# 3. Create a clone of the rebinned 3mb histogram to hold the ratio
rebinned_ratio_eecs = rebinned_eec_3mb.Clone("rebinned_ratio_log")
rebinned_v2_01_ratio = rebinned_eec_01_v2.Clone("rebinned_ratio_v2_01")
rebinned_v2_03_ratio = rebinned_eec_03_v2.Clone("rebinned_ratio_v2_03")

# 4. Divide the rebinned histograms 
rebinned_ratio_eecs.Divide(rebinned_eec_0mb)
rebinned_v2_01_ratio.Divide(rebinned_eec_0mb)
rebinned_v2_03_ratio.Divide(rebinned_eec_0mb)

# make TGraphErrors
tgraph_ratio_3mb = ROOT.TGraphErrors(rebinned_ratio_eecs)
tgraph_ratio_v2_01 = ROOT.TGraphErrors(rebinned_v2_01_ratio)
tgraph_ratio_v2_03 = ROOT.TGraphErrors(rebinned_v2_03_ratio)

# Choosing colours:
tgraph_ratio_3mb.SetLineColorAlpha(colours[0], 0.6)
tgraph_ratio_v2_01.SetLineColorAlpha(colours[3], 0.6)
tgraph_ratio_v2_03.SetLineColorAlpha(colours[2], 0.6)

tgraph_ratio_3mb.SetMarkerColorAlpha(colours[0], 0.7)
tgraph_ratio_v2_01.SetMarkerColorAlpha(colours[3], 0.7)
tgraph_ratio_v2_03.SetMarkerColorAlpha(colours[2], 0.7)

# markers:
tgraph_ratio_3mb.SetMarkerStyle(20)
tgraph_ratio_v2_01.SetMarkerStyle(20)
tgraph_ratio_v2_03.SetMarkerStyle(20)

tgraph_ratio_3mb.SetMarkerSize(0.8)
tgraph_ratio_v2_01.SetMarkerSize(0.8)
tgraph_ratio_v2_03.SetMarkerSize(0.8)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l_r", "rebinned log ratio of EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_ratio_3mb, "PE")
mg.Add(tgraph_ratio_v2_01, "PE")
mg.Add(tgraph_ratio_v2_03, "PE")

# title
mg.SetTitle("Ratio of EECs for inclusive datasets; #Delta R_{L}; Ratio")
mg.Draw("A") 

# reference line
line = ROOT.TLine(0, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# scale axes
canvas.SetLogx(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0, 2)
mg.SetMinimum(0.95)
#mg.SetMaximum(1.03)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)


# Legend setup 
# legend = ROOT.TLegend(0.58, 0.28, 0.78, 0.38)  for just 0.1 v2
legend = ROOT.TLegend(0.28, 0.48, 0.38, 0.58)
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_3mb, "E2C_{3.0mb} / E2C_{0mb}", "PE")
legend.AddEntry(tgraph_ratio_v2_01, "E2C_{v_{2}=0.1} / E2C_{0mb}", "PE")
legend.AddEntry(tgraph_ratio_v2_03, "E2C_{v_{2}=0.3} / E2C_{0mb}", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_rebinned_inclusive_ratio_log_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_rebinned_inclusive_ratio_log_markers_v2.pdf has been created


In [14]:
# ratio, with markers
# log x axis, original bins
# plot injected over 0mb and 3mb over 0mb ratios


ROOT.gDirectory.Clear()

eec_0mb , eec_3mb = initialise(filepath_0mb_inc, filepath_3mb_inc)

dummy , eec_01_v2 = initialise(filepath_0mb_inc, filepath_0mb_01_v2)  

# 3. Create a clone of the 3mb histogram to hold the ratio
ratio_3mb_eecs = eec_3mb.Clone("rebinned_ratio_log")
ratio_v2_eecs = eec_01_v2.Clone("rebinned_ratio_v2")

# 4. Divide the rebinned histograms 
ratio_3mb_eecs.Divide(eec_0mb)
ratio_v2_eecs.Divide(eec_0mb)

# make TGraphErrors
tgraph_ratio_3mb = ROOT.TGraphErrors(ratio_3mb_eecs)
tgraph_ratio_v2 = ROOT.TGraphErrors(ratio_v2_eecs)

# Choosing colours:
tgraph_ratio_3mb.SetLineColorAlpha(colours[0], 0.6)
tgraph_ratio_v2.SetLineColorAlpha(colours[3], 0.6)

tgraph_ratio_3mb.SetMarkerColorAlpha(colours[0], 0.6)
tgraph_ratio_v2.SetMarkerColorAlpha(colours[3], 0.6)

# markers:
tgraph_ratio_3mb.SetMarkerStyle(20)
tgraph_ratio_v2.SetMarkerStyle(20)

tgraph_ratio_3mb.SetMarkerSize(0.8)
tgraph_ratio_v2.SetMarkerSize(0.8)

# initialise canvas
canvas = ROOT.TCanvas("c_eec_l_r", "log ratio of EECs for inclusive datasets", 750, 650)

canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg = ROOT.TMultiGraph()

# Add the data markers
mg.Add(tgraph_ratio_3mb, "PE")
mg.Add(tgraph_ratio_v2, "PE")

# title
mg.SetTitle("Ratio of EECs for inclusive datasets; #Delta R_{L}; Ratio")
mg.Draw("A") 

# reference line
line = ROOT.TLine(0, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# scale axes
canvas.SetLogx(1)
mg.GetXaxis().SetLimits(0.001, 1.0)
mg.GetYaxis().SetLimits(0, 2)
mg.SetMinimum(0.97)
mg.SetMaximum(1.03)

mg.GetXaxis().SetTitleSize(0.045)
mg.GetYaxis().SetTitleSize(0.045)
mg.GetXaxis().SetLabelSize(0.04)
mg.GetYaxis().SetLabelSize(0.04)


# Legend setup
#legend = ROOT.TLegend(0.28, 0.78, 0.48, 0.88) 
legend = ROOT.TLegend(0.58, 0.28, 0.78, 0.38) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_3mb, "E2C_{3.0mb} / E2C_{0mb}", "PE")
legend.AddEntry(tgraph_ratio_v2, "E2C_{v_{2}=0.1} / E2C_{0mb}", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_inclusive_ratio_log_markers_v2.pdf")
canvas.Close()


Info in <TCanvas::Print>: pdf file EEC_inclusive_ratio_log_markers_v2.pdf has been created


# High Nch

## EEC plots

## ratios

In [9]:
#  V2=0.1 and V2=0.3 separately

ROOT.gDirectory.Clear()

# initialise
eec_0mb = initialise_highmult(filepath_0mb_nch60)
eec_3mb = initialise_highmult(filepath_3mb_nch60)
eec_01_v2 = initialise_highmult(filepath_v2_nch60_01) 
eec_03_v2 = initialise_highmult(filepath_v2_nch60_03)  

# rebin
rebinned_hists_0mb = {}
rebinned_hists_3mb = {}
rebinned_hists_01_v2 = {}
rebinned_hists_03_v2 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    rebinned_hists_0mb[bin_key] = rebin_log(eec_0mb[bin_key], f"eec_0mb_log{bin_name}", 50)
    rebinned_hists_3mb[bin_key] = rebin_log(eec_3mb[bin_key], f"eec_3mb_log{bin_name}", 50)
    rebinned_hists_01_v2[bin_key] = rebin_log(eec_01_v2[bin_key], f"eec_01_v2_log{bin_name}", 50)
    rebinned_hists_03_v2[bin_key] = rebin_log(eec_03_v2[bin_key], f"eec_03_v2_log{bin_name}", 50)


# make clones
ratio_hists_3mb = {}
ratio_hists_v2_01 = {}
ratio_hists_v2_03 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    ratio_hists_3mb[bin_key] = rebinned_hists_3mb[bin_key].Clone(f"ratio_3mb_{bin_name}")
    ratio_hists_v2_01[bin_key] = rebinned_hists_01_v2[bin_key].Clone(f"ratio_v2_01_{bin_name}")
    ratio_hists_v2_03[bin_key] = rebinned_hists_03_v2[bin_key].Clone(f"ratio_v2_03_{bin_name}")


# 4. Divide the rebinned histogram clones
for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    ratio_hists_3mb[bin_key].Divide(rebinned_hists_0mb[bin_key])
    ratio_hists_v2_01[bin_key].Divide(rebinned_hists_0mb[bin_key])
    ratio_hists_v2_03[bin_key].Divide(rebinned_hists_0mb[bin_key])


# make TGraphErrors
tgraph_ratio_3mb = {}
tgraph_ratio_v2_01 = {}
tgraph_ratio_v2_03 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    tgraph_ratio_3mb[bin_key] = ROOT.TGraphErrors(ratio_hists_3mb[bin_key])
    tgraph_ratio_v2_01[bin_key] = ROOT.TGraphErrors(ratio_hists_v2_01[bin_key])
    tgraph_ratio_v2_03[bin_key] = ROOT.TGraphErrors(ratio_hists_v2_03[bin_key])

    # set marker style
    tgraph_ratio_3mb[bin_key].SetMarkerStyle(24) #open circle
    tgraph_ratio_v2_01[bin_key].SetMarkerStyle(25) #open square 
    tgraph_ratio_v2_03[bin_key].SetMarkerStyle(26) #open triangle

    # set marker size
    tgraph_ratio_3mb[bin_key].SetMarkerSize(0.4)
    tgraph_ratio_v2_01[bin_key].SetMarkerSize(0.4)
    tgraph_ratio_v2_03[bin_key].SetMarkerSize(0.4)


# Choosing colours:
idx = 0
for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    #line colour
    tgraph_ratio_3mb[bin_key].SetLineColorAlpha(colours[idx], 0.4)
    tgraph_ratio_v2_01[bin_key].SetLineColorAlpha(colours[idx], 0.4)
    tgraph_ratio_v2_03[bin_key].SetLineColorAlpha(colours[idx], 0.4)

    #marker colour
    tgraph_ratio_3mb[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)
    tgraph_ratio_v2_01[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)
    tgraph_ratio_v2_03[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)

    idx+=1 #change colour index




In [13]:

# CANVAS SETTINGS FOR V2=0.1
# initialise canvas
canvas = ROOT.TCanvas("c_eec_r", "ratios of EECs for different multiplicity bins", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg1 = ROOT.TMultiGraph()
mg2 = ROOT.TMultiGraph()
mg3 = ROOT.TMultiGraph()

# Add the data markers 

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    mg1.Add(tgraph_ratio_3mb[bin_key], "PE")
    mg2.Add(tgraph_ratio_v2_01[bin_key], "PE")
    mg3.Add(tgraph_ratio_v2_03[bin_key], "PE")


# title
mg2.SetTitle("EEC ratio for high multiplicity bins (v2=0.1); #Delta R_{L}; E2C_{v_{2}=0.1} / E2C_{0mb}")
mg2.Draw("A") 

# reference line
line = ROOT.TLine(0.001, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Log scale axes
canvas.SetLogx(1)
mg2.GetXaxis().SetLimits(0.001, 1.0)
mg2.GetYaxis().SetLimits(0, 2.0)

mg2.GetXaxis().SetTitleSize(0.045)
mg2.GetYaxis().SetTitleSize(0.045)
mg2.GetXaxis().SetLabelSize(0.04)
mg2.GetYaxis().SetLabelSize(0.04)

mg2.SetMinimum(0.8) # avoiding zeroes due to rebinning
mg2.SetMaximum(1.3)

# Legend setup
legend = ROOT.TLegend(0.28, 0.24, 0.68, 0.44) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_v2_01["60 < Nch < 71"], "60 #leq N_{ch} < 71", "PE")
legend.AddEntry(tgraph_ratio_v2_01["71 < Nch < 78"], "71 #leq N_{ch} < 78", "PE")
legend.AddEntry(tgraph_ratio_v2_01["78 < Nch < 91"], "78 #leq N_{ch} < 91", "PE")
legend.AddEntry(tgraph_ratio_v2_01["91 < Nch < 97"], "91 #leq N_{ch} < 97", "PE")
legend.AddEntry(tgraph_ratio_v2_01["97 < Nch < 1000"], "97 #leq N_{ch} < 1000", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_v2_01_rebinned_plot_ratio_markers.pdf")
canvas.Close()




Info in <TCanvas::Print>: pdf file EEC_v2_01_rebinned_plot_ratio_markers.pdf has been created


In [10]:

# CANVAS SETTINGS FOR V2=0.3
# initialise canvas
canvas = ROOT.TCanvas("c_eec_r", "ratios of EECs for different multiplicity bins", 750, 650)
canvas.SetLeftMargin(0.15)
canvas.SetBottomMargin(0.15)
canvas.SetGrid()

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# initialise multigraph
mg1 = ROOT.TMultiGraph()
mg2 = ROOT.TMultiGraph()
mg3 = ROOT.TMultiGraph()

# Add the data markers 

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    mg1.Add(tgraph_ratio_3mb[bin_key], "PE")
    mg2.Add(tgraph_ratio_v2_01[bin_key], "PE")
    mg3.Add(tgraph_ratio_v2_03[bin_key], "PE")


# title
mg3.SetTitle("EEC ratio for high multiplicity bins (v2=0.3); #Delta R_{L}; E2C_{v_{2}=0.3} / E2C_{0mb}")
mg3.Draw("A") 

# reference line
line = ROOT.TLine(0.001, 1.0, 1.0, 1.0)
line.SetLineStyle(2) # Dashed configuration
line.SetLineColor(ROOT.kBlack)
line.Draw()

# Log scale axes
canvas.SetLogx(1)
mg3.GetXaxis().SetLimits(0.001, 1.0)
mg3.GetYaxis().SetLimits(0, 2.0)

mg3.GetXaxis().SetTitleSize(0.045)
mg3.GetYaxis().SetTitleSize(0.045)
mg3.GetXaxis().SetLabelSize(0.04)
mg3.GetYaxis().SetLabelSize(0.04)

mg3.SetMinimum(0.8) # avoiding zeroes due to rebinning
mg3.SetMaximum(1.3)

# Legend setup
legend = ROOT.TLegend(0.28, 0.24, 0.68, 0.44) 
legend.SetBorderSize(0)
legend.SetFillStyle(0) 
legend.SetTextSize(0.035)
legend.AddEntry(tgraph_ratio_v2_03["60 < Nch < 71"], "60 #leq N_{ch} < 71", "PE")
legend.AddEntry(tgraph_ratio_v2_03["71 < Nch < 78"], "71 #leq N_{ch} < 78", "PE")
legend.AddEntry(tgraph_ratio_v2_03["78 < Nch < 91"], "78 #leq N_{ch} < 91", "PE")
legend.AddEntry(tgraph_ratio_v2_03["91 < Nch < 97"], "91 #leq N_{ch} < 97", "PE")
legend.AddEntry(tgraph_ratio_v2_03["97 < Nch < 1000"], "97 #leq N_{ch} < 1000", "PE")
legend.Draw()

canvas.Update()
canvas.SaveAs("EEC_v2_03_rebinned_plot_ratio_markers.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file EEC_v2_03_rebinned_plot_ratio_markers.pdf has been created


In [ ]:
# attempt at 3 way canvas split
# ratio, with markers
# log x axis, rebinned
# plot injected over 0mb and 3mb over 0mb ratios


ROOT.gDirectory.Clear()

# initialise
eec_0mb = initialise_highmult(filepath_0mb_nch60)
eec_3mb = initialise_highmult(filepath_3mb_nch60)
eec_01_v2 = initialise_highmult(filepath_v2_nch60_01) 
eec_03_v2 = initialise_highmult(filepath_v2_nch60_03)  

# rebin
rebinned_hists_0mb = {}
rebinned_hists_3mb = {}
rebinned_hists_01_v2 = {}
rebinned_hists_03_v2 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    rebinned_hists_0mb[bin_key] = rebin_log(eec_0mb[bin_key], f"eec_0mb_log{bin_name}", 50)
    rebinned_hists_3mb[bin_key] = rebin_log(eec_3mb[bin_key], f"eec_3mb_log{bin_name}", 50)
    rebinned_hists_01_v2[bin_key] = rebin_log(eec_01_v2[bin_key], f"eec_01_v2_log{bin_name}", 50)
    rebinned_hists_03_v2[bin_key] = rebin_log(eec_03_v2[bin_key], f"eec_03_v2_log{bin_name}", 50)


# make clones
ratio_hists_3mb = {}
ratio_hists_v2_01 = {}
ratio_hists_v2_03 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    ratio_hists_3mb[bin_key] = rebinned_hists_3mb[bin_key].Clone(f"ratio_3mb_{bin_name}")
    ratio_hists_v2_01[bin_key] = rebinned_hists_01_v2[bin_key].Clone(f"ratio_v2_01_{bin_name}")
    ratio_hists_v2_03[bin_key] = rebinned_hists_03_v2[bin_key].Clone(f"ratio_v2_03_{bin_name}")


# 4. Divide the rebinned histogram clones
for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    ratio_hists_3mb[bin_key].Divide(rebinned_hists_0mb[bin_key])
    ratio_hists_v2_01[bin_key].Divide(rebinned_hists_0mb[bin_key])
    ratio_hists_v2_03[bin_key].Divide(rebinned_hists_0mb[bin_key])


# make TGraphErrors
tgraph_ratio_3mb = {}
tgraph_ratio_v2_01 = {}
tgraph_ratio_v2_03 = {}

for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    tgraph_ratio_3mb[bin_key] = ROOT.TGraphErrors(ratio_hists_3mb[bin_key])
    tgraph_ratio_v2_01[bin_key] = ROOT.TGraphErrors(ratio_hists_v2_01[bin_key])
    tgraph_ratio_v2_03[bin_key] = ROOT.TGraphErrors(ratio_hists_v2_03[bin_key])

    # set marker style
    tgraph_ratio_3mb[bin_key].SetMarkerStyle(24) #open circle
    tgraph_ratio_v2_01[bin_key].SetMarkerStyle(25) #open square 
    tgraph_ratio_v2_03[bin_key].SetMarkerStyle(26) #open triangle

    # set marker size
    tgraph_ratio_3mb[bin_key].SetMarkerSize(0.4)
    tgraph_ratio_v2_01[bin_key].SetMarkerSize(0.4)
    tgraph_ratio_v2_03[bin_key].SetMarkerSize(0.4)


# Choosing colours:
idx = 0
for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    #line colour
    tgraph_ratio_3mb[bin_key].SetLineColorAlpha(colours[idx], 0.4)
    tgraph_ratio_v2_01[bin_key].SetLineColorAlpha(colours[idx], 0.4)
    tgraph_ratio_v2_03[bin_key].SetLineColorAlpha(colours[idx], 0.4)

    #marker colour
    tgraph_ratio_3mb[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)
    tgraph_ratio_v2_01[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)
    tgraph_ratio_v2_03[bin_key].SetMarkerColorAlpha(colours[idx], 0.7)

    idx+=1 #change colour index



# CANVAS SETTINGS
# initialise canvas
canvas = ROOT.TCanvas("c_eec_l_r", "rebinned log ratio of EECs for inclusive datasets", 3400, 1000)

canvas.Divide(3, 1) # 3 side by side pads

transparent_grid = ROOT.TColor.GetColorTransparent(12, 0.4)
ROOT.gStyle.SetGridColor(transparent_grid)

# Define titles for each subplot
titles = [
    "3.0mb Ratio; #Delta R_{L}; Ratio",
    "v2 = 0.1 Ratio; #Delta R_{L}; Ratio",
    "v2 = 0.3 Ratio; #Delta R_{L}; Ratio"
]

# initialise multigraphs
mg1 = ROOT.TMultiGraph()
mg2 = ROOT.TMultiGraph()
mg3 = ROOT.TMultiGraph()

# Add the data markers
for mult_bin in high_bins:
    bin_key = f"{mult_bin[0]} < Nch < {mult_bin[1]}"
    bin_name = f"{mult_bin[0]}_{mult_bin[1]}"

    mg1.Add(tgraph_ratio_3mb[bin_key], "PE")
    mg2.Add(tgraph_ratio_v2_01[bin_key], "PE")
    mg3.Add(tgraph_ratio_v2_03[bin_key], "PE")

multigraphs = [mg1, mg2, mg3]
graphs_for_legend = [tgraph_ratio_3mb, tgraph_ratio_v2_01, tgraph_ratio_v2_03]
legend_labels = ["60 #leq N_{ch} < 71", "71 #leq N_{ch} < 78", "78 #leq N_{ch} < 91", "91 #leq N_{ch} < 97", "97 #leq N_{ch} < 1000"]

line = ROOT.TLine(0.001, 1.0, 1.0, 1.0)
line.SetLineStyle(2) 
line.SetLineColor(ROOT.kBlack) #redraw on each pad


# loop through pads
for i in range(3):
    pad = canvas.cd(i + 1) # ROOT pads are 1-indexed
    
    # Pad margins and grid
    pad.SetLeftMargin(0.15)
    pad.SetBottomMargin(0.15)
    pad.SetTopMargin(0.18)
    pad.SetGrid()
    pad.SetLogx(1) # Apply log scale to x-axis for this pad

    # Draw multigraph
    mg = multigraphs[i]
    mg.SetTitle(titles[i])
    mg.Draw("A") 
    
    # Scale axes (Fixed: using 'mg' explicitly per pad now)
    mg.GetXaxis().SetLimits(0.001, 1.0)
    mg.SetMinimum(0.6)
    mg.SetMaximum(1.4) 
    
    mg.GetXaxis().SetTitleSize(0.045)
    mg.GetYaxis().SetTitleSize(0.045)
    mg.GetXaxis().SetLabelSize(0.04)
    mg.GetYaxis().SetLabelSize(0.04)
    
    # Draw reference line
    line.Draw()
    
    # Individual Legend Setup per pad
    
    # Add specific entry to this pad's legend
    
    if i == 0: #only adding for left pad
        legend = ROOT.TLegend(0.28, 0.24, 0.68, 0.44) 
        legend.SetBorderSize(0)
        legend.SetFillStyle(0) 
        legend.SetTextSize(0.035)

        legend.AddEntry(tgraph_ratio_v2_01["60 < Nch < 71"], "60 #leq N_{ch} < 71", "PE")
        legend.AddEntry(tgraph_ratio_v2_01["71 < Nch < 78"], "71 #leq N_{ch} < 78", "PE")
        legend.AddEntry(tgraph_ratio_v2_01["78 < Nch < 91"], "78 #leq N_{ch} < 91", "PE")
        legend.AddEntry(tgraph_ratio_v2_01["91 < Nch < 97"], "91 #leq N_{ch} < 97", "PE")
        legend.AddEntry(tgraph_ratio_v2_01["97 < Nch < 1000"], "97 #leq N_{ch} < 1000", "PE")
        
        legend.Draw()
    
        # Keep legend in memory so it doesn't disappear when the loop moves on
        pad.Update()
        #pad.KeepWithCurrentPad(legend)

# Overall Title for the Canvas
canvas.cd(0) # Select main canvas background
title_text = ROOT.TPaveText(0.3, 0.93, 0.7, 0.98, "NDC")
title_text.SetBorderSize(0)
title_text.SetFillStyle(0)
title_text.SetTextSize(0.04)
title_text.SetTextFont(62) # Bold font
title_text.AddText("Ratio of EECs for High Multiplicity Datasets")
title_text.Draw()

# Save output
canvas.Update()
canvas.SaveAs("EEC_3way_rebinned_nch60_ratio_log_markers_v2.pdf")
canvas.Close()

Info in <TCanvas::Print>: pdf file EEC_3way_rebinned_nch60_ratio_log_markers_v2.pdf has been created
